# Loop contributions in the shallow-lake model

## 1. Setup: model, bifurcation diagram

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.integrate import odeint
from scipy.linalg import expm

from tigramite import data_processing as pp
from tigramite import plotting as tp
from tigramite.graphs import Graphs
from tigramite.plotting import plot_time_series_graph
from tigramite.models import LinearMediation

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# from itertools import combinations

colors = px.colors.qualitative.Prism

C = {
    'A_total':    colors[3],
    'A_direct':   colors[4],
    'A_nutrient': colors[8],
    'A_trophic':  colors[2],
    'A_mixed':  colors[10],
    'clear':    colors[6],
    'turbid':   colors[7],
    'L':        colors[7],
    'F':        colors[0],
    'A':        colors[4],
    'M':        colors[6],
    'Z':        colors[2],
    'N':        colors[8],
    'PF':       colors[1],
}

In [ ]:
# ===== Parameters used in the model =====
params = dict(
    # Nutrients
    flush=0.03, c_PA=0.6, c_PM=0.6, eps_N=1e-2,

    # Algae
    g_A=0.45, K_A=5.0, m_A=0.12, c_A=0.021,
    a=0.21, h_A=1.0, eps_A=1e-2,

    # Macrophytes
    g_M=0.375, K_M=1.5, s=3.0,
    m_M=0.03, c_M=0.021, eps_M=1e-2,

    # Zooplankton
    e_Z=0.5, m_Z=0.006, c_Z=0.024, eps_Z=1e-2,

    # Planktivorous fish
    p_F=0.075, e_F=0.7,
    m_F=0.0045, c_F=0.036, eps_PF=1e-2,

    # Macrophyte refuge
    q_ref=2.0,
)


# 7-variable state: L is exogenous nutrient loading, F is exogenous fishing pressure.
state_names = ['L', 'N', 'A', 'M', 'Z', 'PF', 'F']
iL, iN, iA, iM, iZ, iPF, iF = 0, 1, 2, 3, 4, 5, 6

# Local indices for the 5x5 Jacobian / covariance matrices (N,A,M,Z,PF = 0..4)
reduced_state_names = ['N', 'A', 'M', 'Z', 'PF']
jN, jA, jM, jZ, jPF = 0, 1, 2, 3, 4

FAST, SLOW = 1, 2
LAGS = {
    'l_L_N':   0,                     # exogenous
    'l_N_N':   FAST, 'l_M_N': FAST, 'l_A_N': FAST,   # -> N (fast)
    'l_N_A':   FAST, 'l_A_A': FAST, 'l_Z_A': FAST,   # -> A (fast)
    'l_N_M':   SLOW, 'l_A_M': SLOW, 'l_M_M': FAST,   # -> M (slow)
    'l_A_Z':   FAST, 'l_Z_Z': FAST, 'l_PF_Z': FAST, 'l_M_Z':   FAST, # -> Z (fast)
    'l_Z_PF':  SLOW, 'l_PF_PF': FAST, 'l_M_PF':  SLOW,  # -> PF (slow)
    'l_F_PF':  0,                     # exogenous
}

In [ ]:
def model_without_delay(state, L, F, p=params):
    N, A, M, Z, PF = state
    GA  = p['g_A'] * N/(p['K_A'] + N) * A
    GM  = p['g_M'] * N/(p['K_M'] + N) * M / (1.0 + p['s'] * A*A)
    fr  = A / (A + p['h_A'])
    graze = p['a'] * fr * Z
    gain  = p['e_Z'] * p['a'] * fr * Z
    muA = p['m_A']*A + p['c_A']*A*A
    muM = p['m_M']*M + p['c_M']*M*M
    pf_eff = p['p_F'] / (1.0 + p['q_ref']*M)
    dN  = L - p['flush']*N - GA - GM + p['c_PA']*muA + p['c_PM']*muM + p['eps_N']
    dA  = p['eps_A'] + GA    - graze        - muA
    dM  = p['eps_M'] + GM                   - muM
    dZ  = p['eps_Z'] + gain  - pf_eff*Z*PF  - p['m_Z']*Z  - p['c_Z']*Z*Z
    dPF = p['eps_PF']+ p['e_F']*pf_eff*Z*PF - F*PF - p['m_F']*PF - p['c_F']*PF*PF
    return np.array([dN, dA, dM, dZ, dPF])

In [ ]:
def simulate_sdde(state0, p, T, dt=0.1, L_mean=1.0, F_mean=0.0,
                  noise_std=0.01, sigma_L=0, sigma_F=0,
                  seed=0, lags=LAGS, additive_noise=False):
    rng = np.random.default_rng(seed)
    s0 = np.asarray(state0, float); s0_5 = s0[1:6] if len(s0) >= 7 else s0
    L0 = float(np.mean(L_mean)); F0 = float(np.mean(F_mean))
    lags_dt = {k:int(round(v/dt)) for k,v in lags.items()}

    rl = {k:max(v-1,0) for k,v in lags_dt.items()}
    max_lag = max(lags_dt.values())
    hist = np.tile(np.array([L0,*s0_5,F0],float),(max_lag+1,1))
    traj = np.zeros((T,7)); state = s0_5.copy(); sigma = np.full(5,noise_std)
    L_n_arr = (np.full(T,L_mean,float) if np.isscalar(L_mean) else np.asarray(L_mean,float)) + sigma_L*rng.standard_normal(T)
    F_n_arr = (np.full(T,F_mean,float) if np.isscalar(F_mean) else np.asarray(F_mean,float)) + sigma_F*rng.standard_normal(T)

    for n in range(T):
        N, A, M, Z, PF = state
        L_n, F_n = L_n_arr[n], F_n_arr[n]
        traj[n] = [L_n, N, A, M, Z, PF, F_n]
        # Cross-couplings at their (off-by-one-corrected) link lags:
        N_for_A = hist[rl['l_N_A'], iN]
        N_for_M = hist[rl['l_N_M'], iN]         
        A_for_M = hist[rl['l_A_M'], iA]         
        Z_for_A = hist[rl['l_Z_A'], iZ]         
        A_for_Z = hist[rl['l_A_Z'], iA]         
        M_for_N = hist[rl['l_M_N'], iM]         
        PF_for_Z= hist[rl['l_PF_Z'], iPF]       
        Z_for_PF= hist[rl['l_Z_PF'], iZ]        
        M_for_Z = hist[rl['l_M_Z'], iM]         
        M_for_PF= hist[rl['l_M_PF'], iM]        
        N_for_N = hist[rl['l_N_N'], iN]         
        A_for_N = hist[rl['l_A_N'], iA]         
        # Self-loops are all lag 1
        # only the exogenous drivers L, F act at lag 0.
        A_self  = hist[rl['l_A_A'],   iA]
        M_self  = hist[rl['l_M_M'],   iM]
        Z_self  = hist[rl['l_Z_Z'],   iZ]
        PF_self = hist[rl['l_PF_PF'], iPF]
        # dN
        GA_s = p['g_A']*N_for_N/(p['K_A']+N_for_N)*A_for_N
        GM_s = p['g_M']*N_for_N/(p['K_M']+N_for_N)*M_for_N/(1.0+p['s']*A_for_N**2)
        muA_N = p['m_A']*A_for_N + p['c_A']*A_for_N**2
        muM_N = p['m_M']*M_for_N + p['c_M']*M_for_N**2
        dN = L_n - p['flush']*N_for_N - GA_s - GM_s + p['c_PA']*muA_N + p['c_PM']*muM_N + p['eps_N']
        # dA
        GA = p['g_A']*N_for_A/(p['K_A']+N_for_A)*A_self
        graze = p['a']*A_self/(A_self+p['h_A'])*Z_for_A
        muA = p['m_A']*A_self + p['c_A']*A_self**2
        dA = p['eps_A']+ GA - graze - muA
        # dM
        GM = p['g_M']*N_for_M/(p['K_M']+N_for_M)*M_self/(1.0+p['s']*A_for_M**2)
        muM = p['m_M']*M_self + p['c_M']*M_self**2
        dM = p['eps_M']+ GM - muM
        # dZ
        gain = p['e_Z']*p['a']*A_for_Z/(A_for_Z+p['h_A'])*Z_self
        pf_eff_Z = p['p_F']/(1.0+p['q_ref']*M_for_Z)
        dZ = p['eps_Z'] + gain - pf_eff_Z*Z_self*PF_for_Z - p['m_Z']*Z_self - p['c_Z']*Z_self**2
        # dPF
        pf_eff_PF = p['p_F']/(1.0+p['q_ref']*M_for_PF)
        dPF = p['eps_PF']+ p['e_F']*pf_eff_PF*Z_for_PF*PF_self - F_n*PF_self - p['m_F']*PF_self - p['c_F']*PF_self**2
        drift=np.array([dN,dA,dM,dZ,dPF]); eta=rng.standard_normal(5)

        incr = sigma*np.sqrt(dt)*eta if additive_noise else sigma*state*np.sqrt(dt)*eta
        new=state+dt*drift+incr
        hist[1:]=hist[:-1]; hist[0]=[L_n,*new,F_n]; state=new
    return traj

### Equilibria and bifurcation diagram in L

In [ ]:
def find_equilibria_from_initial_conditions(L, F, p, T_integrate=8000, dt=0.1):
    """Relax two initial conditions under the noise-free SDDE to their respective
    equilibria. Returns two 7-element arrays [L, N, A, M, Z, PF, F]."""
    x0_clear  = np.array([L, 2.0, 0.1, 8.0, 0.5, 0.4, F])
    x0_turbid = np.array([L, 5.0, 5.0, 0.1, 1.0, 0.5, F])
    n = int(T_integrate / dt)
    sol_c = simulate_sdde(x0_clear,  p, n, dt=dt, L_mean=L, F_mean=F,
                          noise_std=0.0, sigma_L=0, sigma_F=0)[n//2:].mean(0)
    sol_t = simulate_sdde(x0_turbid, p, n, dt=dt, L_mean=L, F_mean=F,
                          noise_std=0.0, sigma_L=0, sigma_F=0)[n//2:].mean(0)
    return sol_c, sol_t

In [ ]:
def branches_by_continuation(scan_values, which, fixed, _unused, params,
                             t_relax=5000, n_t=400):
    """Trace clear & turbid branches by continuation, seeding each step from the
    previous equilibrium. `which`='L' scans L at fixed F (=fixed); `which`='F' scans
    F at fixed L (=fixed). Undelayed dynamics (equilibria & folds are delay-independent);
    continuation, not a fixed initial condition, is what follows each branch to its true
    fold. Returns (clear, turbid) as (len(scan), 7) arrays [L,N,A,M,Z,PF,F], matching
    find_equilibria_from_initial_conditions' layout so iA/iM indexing is unchanged."""
    sv = np.asarray(scan_values, float)
    t = np.linspace(0, t_relax, n_t)

    def LF(v):
        return (v, fixed) if which == 'L' else (fixed, v)

    def relax(s5, v):
        L, F = LF(v)
        out = odeint(lambda x, tt: model_without_delay(np.array(x), L=L, F=F, p=params),
                     s5, t, mxstep=10000)[-1]
        return out, L, F

    L0, F0 = LF(sv[0]); L1, F1 = LF(sv[-1])
    clear = np.zeros((len(sv), 7))
    s = find_equilibria_from_initial_conditions(L=L0, F=F0, p=params)[0][1:6]
    for i, v in enumerate(sv):                       # forward continuation (clear)
        s, L, F = relax(s, v); clear[i] = [L, *s, F]
    turbid = np.zeros((len(sv), 7))
    s = find_equilibria_from_initial_conditions(L=L1, F=F1, p=params)[1][1:6]
    for i, v in reversed(list(enumerate(sv))):       # backward continuation (turbid)
        s, L, F = relax(s, v); turbid[i] = [L, *s, F]
    return clear, turbid

In [ ]:
L_scan = np.linspace(0.0, 1.7, 80)
F_exogenous = 0.01
clear_branch, turbid_branch = branches_by_continuation(L_scan, 'L', F_exogenous, None, params)

fig_bif_diagram = make_subplots(rows=1, cols=2, shared_xaxes=False, horizontal_spacing=0.15)
arrow_step = 8
for col, var_idx in enumerate([iA, iM], start=1):
    fig_bif_diagram.add_trace(go.Scatter(x=L_scan, y=clear_branch[:, var_idx],
        mode='lines', name='from clear state', line=dict(color=colors[6], dash='dashdot'),
        legendgroup='clear', showlegend=(col == 1)), row=1, col=col)
    fig_bif_diagram.add_trace(go.Scatter(x=L_scan[5::arrow_step], y=clear_branch[5::arrow_step, var_idx],
        mode='markers', marker=dict(symbol='arrow', color=colors[6], size=10, angleref='previous'),
        legendgroup='clear', showlegend=False), row=1, col=col)
    fig_bif_diagram.add_trace(go.Scatter(x=L_scan, y=turbid_branch[:, var_idx],
        mode='lines', name='from turbid state', line=dict(color=colors[2], dash='dot'),
        legendgroup='turbid', showlegend=(col == 1)), row=1, col=col)
    fig_bif_diagram.add_trace(go.Scatter(x=L_scan[5::arrow_step], y=turbid_branch[5::arrow_step, var_idx],
        mode='markers', marker=dict(symbol='arrow', color=colors[2], size=10, angleref='previous', angle=180),
        legendgroup='turbid', showlegend=False), row=1, col=col)
fig_bif_diagram.update_xaxes(title_text='Nutrient loading L')
fig_bif_diagram.update_yaxes(title_text='Algae A', row=1, col=1)
fig_bif_diagram.update_yaxes(title_text='Macrophytes M', row=1, col=2)
fig_bif_diagram.update_layout(height=400, width=800, template='simple_white',
    legend=dict(x=0.22, y=1.15, orientation='h', entrywidth=130))
fig_bif_diagram.show()

In [ ]:
### F_exogenous scan for bifurcation overlay
F_values = [-0.0, 0.01, 0.1]
F_colors = [colors[3], colors[5], colors[7]]
F_dashes = ['dashdot', 'dash', 'dot']
L_scan_F = np.linspace(0.0, 1.7, 80)
arrow_steps = [19, 22, 20]
fig_effect_F_bif_diagram = make_subplots(rows=1, cols=2, horizontal_spacing=0.15)
for f_idx, (F_val, col_hex, dash) in enumerate(zip(F_values, F_colors, F_dashes)):
    cb, tb = branches_by_continuation(L_scan_F, 'L', F_val, None, params)
    sizes = [12 if (i - f_idx) % arrow_steps[f_idx] == 0 else 0 for i in range(len(L_scan_F))]
    fig_effect_F_bif_diagram.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color=col_hex, dash=dash, width=2), name=f'F = {F_val}'))
    for var_col, var_idx in [(1, iA), (2, iM)]:
        fig_effect_F_bif_diagram.add_trace(go.Scatter(x=L_scan_F, y=cb[:, var_idx], mode='lines+markers',
            line=dict(color=col_hex, dash=dash, width=1.5),
            marker=dict(symbol='arrow', color=col_hex, size=sizes, angleref='previous'),
            showlegend=False), row=1, col=var_col)
        fig_effect_F_bif_diagram.add_trace(go.Scatter(x=L_scan_F, y=tb[:, var_idx], mode='lines+markers',
            line=dict(color=col_hex, dash=dash, width=1.5),
            marker=dict(symbol='arrow', color=col_hex, size=sizes, angleref='previous', angle=180),
            showlegend=False), row=1, col=var_col)
fig_effect_F_bif_diagram.update_xaxes(title_text='Nutrient loading L')
fig_effect_F_bif_diagram.update_yaxes(title_text='Algae A', row=1, col=1)
fig_effect_F_bif_diagram.update_yaxes(title_text='Macrophytes M', row=1, col=2)
fig_effect_F_bif_diagram.update_layout(height=400, width=800,
    legend=dict(x=0.14, y=1.15, orientation='h', entrywidth=80), template='simple_white')
fig_effect_F_bif_diagram.show()

In [ ]:
# fig_effect_F_bif_diagram.write_image('figures/bifurcation_diagram_F_effect.svg')

In [ ]:
### Bifurcation diagram in F with fixed L values
L_bif_values = [0.0, 0.9, 1.6]
L_bif_colors = [colors[3], colors[5], colors[7]]
L_bif_dashes = ['dashdot', 'dash', 'dot']
F_bif_scan = np.linspace(-0.5, 0.5, 80)
arrow_steps = [24, 26, 28]
fig_bif_F = make_subplots(rows=1, cols=2, horizontal_spacing=0.15)
for f_idx, (L_val, col_hex, dash) in enumerate(zip(L_bif_values, L_bif_colors, L_bif_dashes)):
    cb, tb = branches_by_continuation(F_bif_scan, 'F', L_val, None, params)
    fig_bif_F.add_trace(go.Scatter(x=[None], y=[None], mode='lines',
        line=dict(color=col_hex, dash=dash, width=2), name=f'L = {L_val}'))
    sizes = [12 if (i - f_idx) % arrow_steps[f_idx] == 0 else 0 for i in range(len(F_bif_scan))]
    for var_col, var_idx in [(1, iA), (2, iM)]:
        fig_bif_F.add_trace(go.Scatter(x=F_bif_scan, y=cb[:, var_idx], mode='lines+markers',
            line=dict(color=col_hex, dash=dash, width=1.5),
            marker=dict(symbol='arrow', color=col_hex, size=sizes, angleref='previous'),
            showlegend=False), row=1, col=var_col)
        fig_bif_F.add_trace(go.Scatter(x=F_bif_scan, y=tb[:, var_idx], mode='lines+markers',
            line=dict(color=col_hex, dash=dash, width=1.5),
            marker=dict(symbol='arrow', color=col_hex, size=sizes, angleref='previous', angle=180),
            showlegend=False), row=1, col=var_col)
fig_bif_F.update_xaxes(title_text='Fishing F')
fig_bif_F.update_yaxes(title_text='Algae A', row=1, col=1)
fig_bif_F.update_yaxes(title_text='Macrophytes M', row=1, col=2)
fig_bif_F.update_layout(height=400, width=800,
    legend=dict(x=0.13, y=1.15, orientation='h', entrywidth=80), template='simple_white')
fig_bif_F.show()

### Visualisation

In [ ]:
L_eq = 0.9; F_eq = 0.01
equilibrium_clear, equilibrium_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

dt = 0.1; sub = int(1/dt)
T_eq = 365*10  # in days
traj_clear  = simulate_sdde(equilibrium_clear,  params, T=int(T_eq/dt), dt=dt, L_mean=L_eq, F_mean=F_eq, noise_std=0.01, seed=0)[::sub]
traj_turbid = simulate_sdde(equilibrium_turbid, params, T=int(T_eq/dt), dt=dt, L_mean=L_eq, F_mean=F_eq, noise_std=0.01, seed=1)[::sub]

In [ ]:
# Plot data
def plot_trajectories(traj, state_names=state_names, title=None):
    fig = make_subplots(rows=1, cols=len(state_names),
                        subplot_titles=[f'{v}' for v in state_names],
                        shared_xaxes=True)
    for i, vname in enumerate(state_names):
        fig.add_trace(go.Scatter(x=np.arange(len(traj)), y=traj[:, i],
                                mode='lines', line=dict(width=0.8, color=C[vname]),
                                showlegend=False),
                    row=1, col=i+1)
    fig.update_layout(height=280, width=240*len(state_names),
                    title_text=title,
                    xaxis=dict(title='Time (days)', range=[0, len(traj)]))
    return fig

def plot_trajectories_stacked(traj, state_names=state_names, title=None):
    n = len(state_names)
    fig = make_subplots(rows=n, cols=1, shared_xaxes=True, vertical_spacing=0.02)
    t = np.arange(len(traj))
    if state_names == reduced_state_names:
        traj = traj[:, 1:]  # skip L, F and I for x-axis if not plotted
    for i, vname in enumerate(state_names):
        fig.add_trace(go.Scatter(x=t, y=traj[:, i], mode='lines',
                                 line=dict(width=0.8, color=C[vname]),
                                 name=vname, showlegend=False),
                      row=i + 1, col=1)
        ax = 'yaxis' if i == 0 else f'yaxis{i + 1}'
        side = 'right' if i % 2 else 'left'
        fig.update_layout(**{ax: dict(side=side, title=vname)})
    fig.update_xaxes(range=[0, len(traj)])
    fig.update_xaxes(title_text='Time (days)', row=n, col=1)
    fig.update_layout(height=110 * n, width=500, title_text=title, plot_bgcolor='rgba(215, 238, 244, 0.3)')
    return fig

In [ ]:
fig_clear = plot_trajectories_stacked(traj_clear, state_names=reduced_state_names, title='Trajectory from clear equilibrium')
fig_turbid = plot_trajectories_stacked(traj_turbid, state_names=reduced_state_names, title='Trajectory from turbid equilibrium')
fig_clear.show()
fig_turbid.show()

## 2. Loop-contribution decomposition

### Analytical decomposition

In [ ]:
def jacobian(state, L, F, p):
    N, A, M, Z, PF = (state[iN], state[iA], state[iM], state[iZ], state[iPF]) if len(state) >= 7 \
                     else (state[0], state[1], state[2], state[3], state[4])
    fN_A = N/(p['K_A']+N); dfN_A = p['K_A']/(p['K_A']+N)**2
    fN_M = N/(p['K_M']+N); dfN_M = p['K_M']/(p['K_M']+N)**2
    sig = 1.0/(1.0+p['s']*A*A); dsig = -p['s']*2*A*sig**2
    dGA_dN = p['g_A']*dfN_A*A; dGA_dA = p['g_A']*fN_A
    dGM_dN = p['g_M']*dfN_M*M*sig; dGM_dM = p['g_M']*fN_M*sig; dGM_dA = p['g_M']*fN_M*M*dsig
    dmuA = p['m_A'] + 2*p['c_A']*A; dmuM = p['m_M'] + 2*p['c_M']*M
    # Holling II grazing derivatives
    h = p['h_A']; fr = A/(A+h); dfr = h/(A+h)**2
    dGr_dA = p['a']*dfr*Z          # d(graze)/dA
    dGr_dZ = p['a']*fr             # d(graze)/dZ
    # macrophyte refuge: pf_eff = p_F/(1+q_ref*M)
    pf = p['p_F']/(1.0+p['q_ref']*M)
    dpf_dM = -p['p_F']*p['q_ref']/(1.0+p['q_ref']*M)**2
    J = np.zeros((5,5))
    J[jN,jN] = -p['flush'] - dGA_dN - dGM_dN
    J[jN,jA] = -dGA_dA - dGM_dA + p['c_PA']*dmuA
    J[jN,jM] = -dGM_dM + p['c_PM']*dmuM
    J[jA,jN] = dGA_dN
    J[jA,jA] = dGA_dA - dGr_dA - dmuA
    J[jA,jZ] = -dGr_dZ
    J[jM,jN] = dGM_dN; J[jM,jA] = dGM_dA; J[jM,jM] = dGM_dM - dmuM
    J[jZ,jA] = p['e_Z']*dGr_dA
    J[jZ,jZ] = p['e_Z']*dGr_dZ - pf*PF - p['m_Z'] - 2*p['c_Z']*Z
    J[jZ,jPF] = -pf*Z
    J[jZ,jM] = -dpf_dM*Z*PF 
    J[jPF,jZ] = p['e_F']*pf*PF
    J[jPF,jPF] = p['e_F']*pf*Z - F - p['m_F'] - 2*p['c_F']*PF
    J[jPF,jM] = p['e_F']*dpf_dM*Z*PF
    return J

In [ ]:
def decomposition_analytical_lagged(state, L, F, params, T_max=30, lags=LAGS, dt=0.1):

    J   = jacobian(state, L, F, params)             # 5x5 continuous Jacobian (per-day rates)
    sub = int(round(1.0 / dt))                 # fine steps per day (=10 for dt=0.1)
    ts  = np.arange(T_max + 1)

    # Row i = destination equation, col j = source variable; order (N, A, M, Z, PF).
    Ld = lags
    LAG_DAYS = np.array([
        [Ld['l_N_N'],  Ld['l_A_N'],  Ld['l_M_N'],  0,            0            ],  # dN
        [Ld['l_N_A'],  Ld['l_A_A'],  0,            Ld['l_Z_A'],  0            ],  # dA
        [Ld['l_N_M'],  Ld['l_A_M'],  Ld['l_M_M'],  0,            0            ],  # dM
        [0,            Ld['l_A_Z'],  Ld['l_M_Z'],  Ld['l_Z_Z'],  Ld['l_PF_Z'] ],  # dZ
        [0,            0,            Ld['l_M_PF'], Ld['l_Z_PF'], Ld['l_PF_PF']],  # dPF
    ], dtype=float)

    # Day-lag -> fine-step offset, using simulate_sdde's off-by-one read correction
    # rl = max(lag_steps - 1, 0) so a d-day link lands exactly d days back after
    # daily subsampling.
    LAG_STEPS = np.maximum(np.round(LAG_DAYS / dt).astype(int) - 1, 0)
    D    = int(LAG_STEPS.max()) + 1            # history depth needed, in fine steps
    cols = np.arange(5)

    def self_response(seed, blocked):
        """Daily seed->seed self-response. Variables in `blocked` are removed from the
        dynamics (rows & cols of J zeroed => every path through them is deleted),
        seeded by a unit impulse in `seed` at day 0 with zero history before it."""
        Jb = J.copy()
        if blocked:
            b = list(blocked)
            Jb[b, :] = 0.0
            Jb[:, b] = 0.0
        buf = np.zeros((D, 5))                 # buf[k] = deviation k fine steps ago
        buf[0, seed] = 1.0                     # unit impulse at t = 0
        g = np.zeros(T_max + 1)
        g[0] = buf[0, seed]                    # = 1
        for n in range(1, T_max * sub + 1):
            read = buf[LAG_STEPS, cols[None, :]]   # read[i,j] = buf[LAG_STEPS[i,j], j]
            incr = (Jb * read).sum(axis=1)         # linearized d/dt of each variable
            new  = buf[0] + dt * incr              # explicit Euler step
            buf  = np.roll(buf, 1, axis=0)         # shift history, drop oldest
            buf[0] = new
            if n % sub == 0:                        # record once per day
                g[n // sub] = buf[0, seed]
        return g

    # A self-response and its loop partition
    Total          = self_response(jA, set())             # all paths
    Nutrient_loose = self_response(jA, {jZ, jPF})          # keep N,M routes
    Trophic_loose  = self_response(jA, {jM, jN})           # keep Z,PF routes
    Direct         = self_response(jA, {jN, jM, jZ, jPF})  # A self-decay only
    Nutrient = Nutrient_loose - Direct
    Nutrient_comp = self_response(jA, {jM, jZ, jPF}) - Direct   # N-only -> A->N->A family
    Nutrient_rel  = Nutrient - Nutrient_comp                    #          A->M->N->A family
    Trophic  = Trophic_loose  - Direct                         # Z,PF-only
    Mixed    = Total - Direct - Nutrient - Trophic

    A_eq = state[iA] if len(state) >= 7 else state[jA]
    return {
        'ts':         ts,
        'A_total':    Total,
        'A_direct':   Direct,
        'A_nutrient': Nutrient,
        'A_trophic':  Trophic,
        'A_mixed':    Mixed,
        'A_nut_competition': Nutrient_comp,
        'A_nut_release':     Nutrient_rel,
        'A_eq':       A_eq,
        'J':          J,
    }

### Validation of the linear approximation

In [ ]:
def delayed_jacobian_blocks(state, L, F, p, lags=LAGS):
    """Partition the Jacobian of the SDDE by lag value.

    Returns {lag: 5x5 matrix}, one block per distinct lag (0, FAST, SLOW).
    Block B[k] holds the partial derivatives of each equation with respect to
    the variables read k days in the past.  Summing all blocks gives the
    conventional (instantaneous) Jacobian.

    Used by simulate_lna_delayed to set up the linearised delay system
    d(dx) = [J0 dx(t) + J1 dx(t-1) + J2 dx(t-2)] dt + noise,
    which is the basis for the LNA around a given equilibrium."""
    N, A, M, Z, PF = (state[iN],state[iA],state[iM],state[iZ],state[iPF]) if len(state)==7 \
                     else (state[0],state[1],state[2],state[3],state[4])
    fN_A=N/(p['K_A']+N); dfN_A=p['K_A']/(p['K_A']+N)**2
    fN_M=N/(p['K_M']+N); dfN_M=p['K_M']/(p['K_M']+N)**2
    sig=1.0/(1.0+p['s']*A*A); dsig=-p['s']*2*A*sig**2
    dmuA=p['m_A']+2*p['c_A']*A; dmuM=p['m_M']+2*p['c_M']*M
    h=p['h_A']; fr=A/(A+h); dfr=h/(A+h)**2
    L_AA=int(lags['l_A_A']); L_MM=int(lags['l_M_M']); L_ZZ=int(lags['l_Z_Z']); L_PFPF=int(lags['l_PF_PF'])
    L0=int(lags['l_N_N']); L_AN=int(lags['l_A_N']); L_NA=int(lags['l_N_A']); L_ZA=int(lags['l_Z_A'])
    L_NM=int(lags['l_N_M']); L_AM=int(lags['l_A_M']); L_MN=int(lags['l_M_N'])
    L_AZ=int(lags['l_A_Z']); L_ZPF=int(lags['l_Z_PF']); L_PFZ=int(lags['l_PF_Z'])
    L_MZ=int(lags['l_M_Z']); L_MPF=int(lags['l_M_PF'])
    pf=p['p_F']/(1.0+p['q_ref']*M); dpf_dM=-p['p_F']*p['q_ref']/(1.0+p['q_ref']*M)**2
    B={}
    def add(lag,i,j,v):
        B.setdefault(lag,np.zeros((5,5)))[i,j]+=v
    # dN row
    add(L0,  jN,jN,-p['flush']-p['g_A']*dfN_A*A-p['g_M']*dfN_M*M*sig)
    add(L_AN,jN,jA,-p['g_A']*fN_A-p['g_M']*fN_M*M*dsig+p['c_PA']*dmuA)
    add(L_MN,jN,jM,-p['g_M']*fN_M*sig+p['c_PM']*dmuM)
    # dA row
    add(L_AA,jA,jA, p['g_A']*fN_A - p['a']*dfr*Z - dmuA)
    add(L_NA,jA,jN, p['g_A']*dfN_A*A)
    add(L_ZA,jA,jZ, -p['a']*fr)
    # dM row
    add(L_MM,jM,jM, p['g_M']*fN_M*sig - dmuM)
    add(L_NM,jM,jN, p['g_M']*dfN_M*M*sig)
    add(L_AM,jM,jA, p['g_M']*fN_M*M*dsig)
    # dZ row
    add(L_ZZ, jZ,jZ, p['e_Z']*p['a']*fr - pf*PF - p['m_Z'] - 2*p['c_Z']*Z)
    add(L_AZ, jZ,jA, p['e_Z']*p['a']*dfr*Z)
    add(L_PFZ,jZ,jPF,-pf*Z)
    add(L_MZ, jZ,jM, -dpf_dM*Z*PF)                    
    # dPF row
    add(L_PFPF,jPF,jPF, p['e_F']*pf*Z - F - p['m_F'] - 2*p['c_F']*PF)
    add(L_ZPF,jPF,jZ,  p['e_F']*pf*PF)
    add(L_MPF,jPF,jM,  p['e_F']*dpf_dM*Z*PF)    
    return B


def simulate_lna_delayed(state0, p, T, dt=0.1, L=1.0, F=0.0,
                         noise_std=0.01, seed=0):
    """Linear delay SDE around x*: dδx = [J_0 δx(t) + J_1 δx(t-1) + J_2 δx(t-2)] dt
    + diag(σ x*) dW."""
    rng   = np.random.default_rng(seed)
    sigma = np.full(5, noise_std) if np.isscalar(noise_std) else np.asarray(noise_std)
    _s = np.asarray(state0)
    xstar = np.array(_s[1:6] if len(_s) >= 7 else _s, float)
    B = delayed_jacobian_blocks(xstar, L, F, p)
    J0 = B.get(0, np.zeros((5,5)))
    J1 = B.get(FAST, np.zeros((5,5)))
    J2 = B.get(SLOW, np.zeros((5,5)))
    k1, k2 = int(round(FAST / dt)), int(round(SLOW / dt))
    max_lag = max(k1, k2)
    hist = np.zeros((max_lag + 1, 5))
    dx   = np.zeros(5)
    traj = np.zeros((T, 5))
    diff_amp = sigma * xstar * np.sqrt(dt)
    for n in range(T):
        traj[n] = xstar + dx
        drift = J0 @ dx + J1 @ hist[k1-1] + J2 @ hist[k2-1]   # off-by-one: read one step earlier
        eta   = rng.standard_normal(5)
        dx    = dx + dt * drift + diff_amp * eta
        hist[1:] = hist[:-1]
        hist[0]  = dx
    return traj


noise_std = 0.01
T_sim_days = 365*10 
dt_sim = 0.1
sub_daily = int(1/dt_sim)
n_lna_sims = 50
L_eq = 0.9; F_eq = 0.01
equilibrium_clear, equilibrium_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

def _visual_run(eq, seed_truth, seed_lna_start):
    _dyn = np.array([iN, iA, iM, iZ, iPF])
    truth = simulate_sdde(eq, params, T=int(T_sim_days/dt_sim), dt=dt_sim,
                            L_mean=L_eq, F_mean=F_eq,
                            noise_std=noise_std, seed=seed_truth)[::sub_daily][:, _dyn]
    sim_list = []
    for k in range(n_lna_sims):
        one_sim = simulate_lna_delayed(eq, params, T=int(T_sim_days/dt_sim), dt=dt_sim,
                                    L=L_eq, F=F_eq, noise_std=noise_std,
                                    seed=seed_lna_start + k)
        sim_list.append(one_sim[::sub_daily])

    sims = np.stack(sim_list)  # (n_lna_sims, T_vis_days, 5)
    return truth, sims


truth_c, sims_c = _visual_run(equilibrium_clear,  seed_truth=991, seed_lna_start=500)
truth_t, sims_t = _visual_run(equilibrium_turbid, seed_truth=992, seed_lna_start=600)

In [ ]:
t_axis = np.arange(truth_c.shape[0])
fig_vis = make_subplots(
    rows=2, cols=2,
    subplot_titles=['A: clear',  'M: clear',
                    'A: turbid', 'M: turbid'],
    shared_xaxes=True,
)
for row, (truth, sims) in enumerate([(truth_c, sims_c), (truth_t, sims_t)], start=1):
    for col, idx in enumerate([jA, jM], start=1):
        leg = (row == 1 and col == 1)
        for k in range(n_lna_sims):
            fig_vis.add_trace(
                go.Scatter(x=t_axis, y=sims[k, :, idx], mode='lines',
                           line=dict(color='rgba(218, 165, 32, 0.18)', width=0.6),
                           name='Linearized simulations',
                           showlegend=(leg and k == 0),
                           legendgroup='lna_delayed'),
                row=row, col=col)
        fig_vis.add_trace(
            go.Scatter(x=t_axis, y=truth[:, idx], mode='lines',
                       line=dict(color=C['L'], width=1.5),
                       name='nonlinear SDDE', showlegend=leg,
                       legendgroup='sdde'),
            row=row, col=col)
fig_vis.update_layout(
    height=500, width=800,
    template='plotly_white',
    legend=dict(x=0.2, y=0.6, orientation='h', entrywidth=160),)
fig_vis.update_xaxes(title_text='Time (days)', row=2, col=1)
fig_vis.update_xaxes(title_text='Time (days)', row=2, col=2)
fig_vis.show()
# fig_vis.write_image('figures/validation_linearization_trajectories_L09.svg')

### Data-driven decomposition

In [ ]:
def make_parents_dict(lags=LAGS):
    n = lambda k: -int(round(lags[k]))
    parents = {
        iL: [], iF: [],
        iN:  [(iN, n('l_N_N')), (iM, n('l_M_N')), (iA, n('l_A_N')), (iL, n('l_L_N')), 
              ],        
        iA:  [(iA, n('l_A_A')),   (iN, n('l_N_A')),  (iZ, n('l_Z_A')), 
              ],
        iM:  [(iM, n('l_M_M')),   (iN, n('l_N_M')),  (iA, n('l_A_M')), 
              ],
        iZ:  [(iZ, n('l_Z_Z')),   (iA, n('l_A_Z')),  (iPF, n('l_PF_Z')), (iM, n('l_M_Z')),
              ],
        iPF: [(iPF, n('l_PF_PF')),(iZ, n('l_Z_PF')), (iF, n('l_F_PF')), (iM, n('l_M_PF')), 
              ],
    }
    return parents

In [ ]:
parents = make_parents_dict()
graph = Graphs.get_graph_from_dict(parents, tau_max=5)
plot_time_series_graph(
    graph=graph, 
    var_names=state_names,
    figsize=(10, 8),
    node_size=0.05,
    #label_fontsize=20,
)
plt.show()

In [ ]:
all_parents = make_parents_dict()

In [ ]:
def causaleffect_decomposition(traj, all_parents=all_parents, tau_max=30):
    # Work on the 5 dynamic variables only (N,A,M,Z,PF).
    # L and F are exogenous drivers: they have no causal parents and only
    # lag-0 links (L->N, F->PF) that fall outside the lagged VAR
    # framework, so we exclude them here.
    _dyn_global = [iN, iA, iM, iZ, iPF]
    _dyn_local  = {iN: jN, iA: jA, iM: jM, iZ: jZ, iPF: jPF}
    _names5 = ['N', 'A', 'M', 'Z', 'PF']
    traj5 = traj[:, _dyn_global] if traj.shape[1] >= 7 else traj
    df = pp.DataFrame(traj5, var_names=_names5)
    # Remap parents dict to local 5-var indices, dropping L, F and I parents
    parents5 = {}
    for g_idx, l_idx in _dyn_local.items():
        parents5[l_idx] = [
            (_dyn_local[p_g], lag)
            for (p_g, lag) in all_parents[g_idx]
            if p_g in _dyn_local  # skip iL, iF
        ]
    med = LinearMediation(dataframe=df)
    med.fit_model(all_parents=parents5, tau_max=tau_max)
    ts = np.arange(tau_max + 1)
    A_total    = np.zeros(tau_max + 1)
    A_direct   = np.zeros(tau_max + 1)
    A_Nutrient = np.zeros(tau_max + 1)
    A_Trophic  = np.zeros(tau_max + 1)
    A_Mixed    = np.zeros(tau_max + 1)
    A_nut_competition = np.zeros(tau_max + 1)   # via M channel
    A_nut_release  = np.zeros(tau_max + 1)   # release family      A->M->N->A (M-involving)

   
    for tau in range(1, tau_max + 1):
        # Total causal effect
        A_total[tau] = med.get_ce(i=jA, tau=-tau, j=jA)

        # Direct effect: paths that exclude any named mediators
        A_direct[tau] = med.get_conditional_mce(i=jA, tau=-tau, j=jA,
                                                k=[jA], notk=[jM, jN, jZ, jPF])
        
        # Loop contributions
        Nut_loose = med.get_conditional_mce(i=jA, tau=-tau, j=jA, k=[jA, jM, jN], notk=[jZ, jPF])
        Tro_loose = med.get_conditional_mce(i=jA, tau=-tau, j=jA, k=[jA, jZ, jPF], notk=[jM, jN])
        Nch_loose = med.get_conditional_mce(i=jA, tau=-tau, j=jA, k=[jA, jN],     notk=[jM, jZ, jPF])

        A_Nutrient[tau]        = Nut_loose - A_direct[tau]
        A_Trophic[tau]         = Tro_loose - A_direct[tau]
        A_nut_competition[tau] = Nch_loose - A_direct[tau]
        A_nut_release[tau]     = A_Nutrient[tau] - A_nut_competition[tau]
        A_Mixed[tau]           = A_total[tau] - A_direct[tau] - A_Nutrient[tau] - A_Trophic[tau]

        result = {
            'ts':         ts,
            'A_total':    A_total,
            'A_direct':   A_direct,
            'A_nutrient': A_Nutrient,
            'A_trophic':  A_Trophic,
            'A_mixed':    A_Mixed,
            'A_nut_competition': A_nut_competition,
            'A_nut_release':     A_nut_release,
        }
    return result

## 3. Loop contributions at the two equilibria

In [ ]:
T_max = 150

In [ ]:
L_eq = 0.9
F_eq = 0.01

n = int(8000/dt)
occ_clear, occ_turbid = find_equilibria_from_initial_conditions(L=L_eq, F=F_eq, p=params)

In [ ]:
dt = 0.1; sub = int(1/dt); noise_std=0.01
burnin = 365*1; T_long = 365*10  # in days
traj_c_long = simulate_sdde(occ_clear,  params, int((burnin+T_long)/dt), dt=dt, L_mean=L_eq, F_mean=F_eq, noise_std=noise_std, sigma_L=0, sigma_F=0, seed=3)[::sub][burnin:]
traj_t_long = simulate_sdde(occ_turbid, params, int((burnin+T_long)/dt), dt=dt, L_mean=L_eq, F_mean=F_eq, noise_std=noise_std, sigma_L=0, sigma_F=0, seed=1)[::sub][burnin:]

In [ ]:
figC = plot_trajectories_stacked(traj_c_long, reduced_state_names, title='Clear')
figT = plot_trajectories_stacked(traj_t_long, reduced_state_names, title='Turbid')
figC.show()
figT.show()

In [ ]:
def detrend_traj(traj, todetrend=[iN, iA, iM, iZ, iPF], method='linear', window=100):
    out = traj.copy()
    dyn_cols = todetrend  # only detrend the specified columns; L and F are constant exogenous inputs
    for i in dyn_cols:
        if method == 'linear':
            # fit a degree-1 polynomial to col i and subtract it, removing the linear trend
            out[:, i] = traj[:, i] - np.polyval(np.polyfit(np.arange(len(traj)), traj[:, i], 1), np.arange(len(traj)))
        elif method == 'moving_average':
            out[:, i] = traj[:, i] - pd.Series(traj[:, i]).rolling(window=window, center=True, min_periods=1).mean().values
    return out

In [ ]:
det_traj_clear = detrend_traj(traj_c_long)
det_traj_turbid = detrend_traj(traj_t_long)

In [ ]:
fig_traj_detrended_clear = plot_trajectories_stacked(det_traj_clear, reduced_state_names, title='Clear equilibrium')
fig_traj_detrended_turbid = plot_trajectories_stacked(det_traj_turbid, reduced_state_names, title='Turbid equilibrium')
fig_traj_detrended_clear.show()
fig_traj_detrended_turbid.show()

In [ ]:
analytical_decomposition_clear = decomposition_analytical_lagged(equilibrium_clear, L_eq, F_eq, params, T_max=T_max, lags=LAGS, dt=dt)
analytical_decomposition_turbid = decomposition_analytical_lagged(equilibrium_turbid, L_eq, F_eq, params, T_max=T_max, lags=LAGS, dt=dt)

In [ ]:
causaleffects_decomposition_clear = causaleffect_decomposition(det_traj_clear, tau_max=T_max)
causaleffects_decomposition_turbid = causaleffect_decomposition(det_traj_turbid, tau_max=T_max)

In [ ]:
def add_A_total_direct(fig, analytical_decomposition, causaleffects_decomposition, row, col):
    """A panel: total + direct only."""
    ts = analytical_decomposition['ts']; ts_tg = causaleffects_decomposition['ts']
    show_leg = (row == 1)
    fig.add_trace(go.Scatter(x=ts, y=analytical_decomposition['A_total'], mode='lines',
                             line=dict(color=C['A_total'], width=2.5),
                             name='Total (analytical)', legendgroup='Atot',
                             showlegend=show_leg),
                  row=row, col=col)
    fig.add_trace(go.Scatter(x=ts, y=analytical_decomposition['A_direct'], mode='lines',
                             line=dict(color=C['A_direct'], width=1.8),
                             name='Direct (analytical)', legendgroup='Adir',
                             showlegend=show_leg),
                  row=row, col=col)
    fig.add_trace(go.Scatter(x=ts_tg, y=causaleffects_decomposition['A_total'], mode='lines+markers',
                             marker=dict(color=C['A_total'], size=3, symbol='circle-open'),
                             line=dict(color=C['A_total'], width=0.5),
                             name='Total (from data)', legendgroup='Atot_t',
                             showlegend=show_leg),
                  row=row, col=col)
    fig.add_trace(go.Scatter(x=ts_tg, y=causaleffects_decomposition['A_direct'], mode='lines+markers',
                             marker=dict(color=C['A_direct'], size=3, symbol='circle-open'),
                             line=dict(color=C['A_direct'], width=0.5),
                             name='Direct (from data)', legendgroup='Adir_t',
                             showlegend=show_leg),
                  row=row, col=col)

def add_A_loops(fig, analytical_decomposition, causaleffects_decomposition, row, col):
    """A panel: Nutrient + Trophic + Mixed (clean partition residuals)."""
    ts = analytical_decomposition['ts']; ts_tg = causaleffects_decomposition['ts']
    show_leg = (row == 1)
    # Analytical lines
    fig.add_trace(go.Scatter(x=ts, y=analytical_decomposition['A_nutrient'], mode='lines',
                             line=dict(color=C['A_nutrient'], width=1.8),
                             name='Nutrient (analytical)', legendgroup='Nut',
                             showlegend=show_leg),
                  row=row, col=col)
    fig.add_trace(go.Scatter(x=ts, y=analytical_decomposition['A_trophic'], mode='lines',
                             line=dict(color=C['A_trophic'], width=1.8),
                             name='Trophic (analytical)', legendgroup='Tro',
                             showlegend=show_leg),
                  row=row, col=col)
    fig.add_trace(go.Scatter(x=ts, y=analytical_decomposition['A_mixed'], mode='lines',
                             line=dict(color=C['A_mixed'], width=1.8),
                             name='Mixed (analytical)', legendgroup='Mix',
                             showlegend=show_leg),
                  row=row, col=col)
    # Tigramite markers
    fig.add_trace(go.Scatter(x=ts_tg, y=causaleffects_decomposition['A_nutrient'], mode='lines+markers',
                             marker=dict(color=C['A_nutrient'], size=3, symbol='circle-open'),
                             line=dict(color=C['A_nutrient'], width=0.5),
                             name='Nutrient (from data)', legendgroup='Nut_t',
                             showlegend=show_leg),
                  row=row, col=col)
    fig.add_trace(go.Scatter(x=ts_tg, y=causaleffects_decomposition['A_trophic'], mode='lines+markers',
                             marker=dict(color=C['A_trophic'], size=3, symbol='circle-open'),
                             line=dict(color=C['A_trophic'], width=0.5),
                             name='Trophic (from data)', legendgroup='Tro_t',
                             showlegend=show_leg),
                  row=row, col=col)
    fig.add_trace(go.Scatter(x=ts_tg, y=causaleffects_decomposition['A_mixed'], mode='lines+markers',
                             marker=dict(color=C['A_mixed'], size=3, symbol='circle-open'),
                             line=dict(color=C['A_mixed'], width=0.5),
                             name='Mixed (from data)', legendgroup='Mix_t',
                             showlegend=show_leg),
                  row=row, col=col)

In [ ]:
# plot results at both equilibria, analytical vs data-driven decomposition
A_column_widths = [0.40, 0.60]   # [total&direct, loops]

fig_A_both_states = make_subplots(rows=2, cols=2,
                     subplot_titles=['A: clear',
                                     'A: clear',
                                     'A: turbid',
                                     'A: turbid'],
                     shared_xaxes=True,
                     column_widths=A_column_widths,
                     horizontal_spacing=0.10, vertical_spacing=0.12)
add_A_total_direct(fig_A_both_states, analytical_decomposition_clear,  causaleffects_decomposition_clear,  row=1, col=1)
add_A_loops       (fig_A_both_states, analytical_decomposition_clear,  causaleffects_decomposition_clear,  row=1, col=2)
add_A_total_direct(fig_A_both_states, analytical_decomposition_turbid, causaleffects_decomposition_turbid, row=2, col=1)
add_A_loops       (fig_A_both_states, analytical_decomposition_turbid, causaleffects_decomposition_turbid, row=2, col=2)
for r in [1, 2]:
    for c in [1, 2]:
        fig_A_both_states.add_hline(y=0, line=dict(color='gray', width=0.5), row=r, col=c)
fig_A_both_states.update_xaxes(title_text='T (days)', row=2, range=[0, 30*5])
fig_A_both_states.update_yaxes(title_text='Loop contribution', col=1)
fig_A_both_states.update_layout(height=720, width=1100,
                   legend=dict(font=dict(size=10)), template='plotly_white')
fig_A_both_states.show()


## 4. Loop contributions along forward and backward sweeps

In [ ]:
T_max = 100

In [ ]:
def sweep_simulate(L_schedule_daily, x0, F, p, window_len=100, step=50,
                   noise_std=0.01, seed=0, dt=0.1, additive_noise=False):
    sub = int(round(1.0 / dt))
    L_schedule_fine = np.repeat(L_schedule_daily, sub)          # piecewise-constant per day
    traj_fine = simulate_sdde(x0, p, len(L_schedule_fine), dt=dt,
                              L_mean=L_schedule_fine, F_mean=F,
                              noise_std=noise_std, seed=seed, additive_noise=additive_noise)
    traj     = traj_fine[::sub]                                 # daily, raw

    T_daily = len(traj)
    n = window_len
    starts = list(range(0, T_daily - n + 1, step))
    centers     = np.array([s + n // 2 for s in starts])
    mean_states = np.array([traj[s:s+n].mean(axis=0) for s in starts])
    mean_Ls     = np.array([L_schedule_daily[s:s+n].mean() for s in starts])
    return dict(traj=traj, L_schedule=np.asarray(L_schedule_daily),
                starts=starts, window_len=n, step=step, centers=centers,
                mean_states=mean_states, mean_Ls=mean_Ls, F=F)

def sweep_decompose_analytical(sweep, params, T_max=20):
    F = sweep['F']
    analytical_decomp = []
    for mean_state, mean_L in zip(sweep['mean_states'], sweep['mean_Ls']):
        # dt=1.0: the analytical decomposition's lag axis is in days
        dec = decomposition_analytical_lagged(mean_state, mean_L, F, params, dt=0.1, T_max=T_max)
        analytical_decomp.append(dec)
    return analytical_decomp

def sweep_decompose_causaleffect(sweep, T_max=20, detrend_window=None):
    win_len = sweep['window_len']
    det_win = detrend_window if detrend_window is not None else win_len
    ce_decomp = []
    for start in sweep['starts']:
        win_raw = sweep['traj'][start:start + win_len]
        window_det = detrend_traj(win_raw, method='linear', window=det_win)
        causal_decomp = causaleffect_decomposition(window_det, tau_max=T_max)
        ce_decomp.append(causal_decomp)
    return ce_decomp


In [ ]:
from joblib import Parallel, delayed

def sweep_decompose_causaleffect(sweep, T_max=20, n_jobs=-1, detrend_window=None):
    win_len = sweep['window_len']
    det_win = detrend_window if detrend_window is not None else win_len
    wins = [
        detrend_traj(sweep['traj'][s:s + win_len], method='linear', window=det_win)
        for s in sweep['starts']
    ]
    return Parallel(n_jobs=n_jobs)(
        delayed(causaleffect_decomposition)(w, tau_max=T_max)
        for w in wins)

In [ ]:
def build_trajectory_figure(sweep_fw, sweep_bw):

    L_fw, traj_fw = sweep_fw['L_schedule'], sweep_fw['traj']
    L_bw, traj_bw = sweep_bw['L_schedule'], sweep_bw['traj']

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=['Trajectory: forward (L ↑)', 'Trajectory: backward (L ↓)'],
        shared_xaxes=True,
        horizontal_spacing=0.12
    )

    for col, (L_arr, traj) in enumerate([(L_fw, traj_fw), (L_bw, traj_bw)], start=1):
        fig.add_trace(
            go.Scatter(x=L_arr, y=traj[:, iA], mode='lines', line=dict(color=C['A'], width=0.6), name='Algae', legendgroup='A', showlegend=(col == 1)), row=1, col=col)
        fig.add_trace(
            go.Scatter(x=L_arr, y=traj[:, iM], mode='lines', line=dict(color=C['M'], width=0.6), name='Macrophytes', legendgroup='M', showlegend=(col == 1)), row=1, col=col)

    fig.update_xaxes(title_text='L')
    fig.update_yaxes(title_text='biomass')
    fig.update_layout(height=330, width=1000, legend=dict(font=dict(size=10)), template='plotly_white')

    return fig

In [ ]:
window_length = 1000
step_size = 10

F_sweep = 0.01
T_max = 100
noise_std = 0.01

# Forward sweep
T_fw = 365*50
L_fw = np.linspace(0.0, 1.5, T_fw)
print('increase of L in one window:', 1.5 * window_length / T_fw)
print('increase of L in one step:', 1.5 * step_size / T_fw)

x0_clear_seed = find_equilibria_from_initial_conditions(L=0, F=F_sweep, p=params)[0]
sweep_fw = sweep_simulate(L_fw, x0_clear_seed, F_sweep, params,
                          window_len=window_length, step=step_size, noise_std=noise_std, seed=8, dt=0.1, additive_noise=False)

# Backward sweep
T_bw = 365*50
L_bw = np.linspace(1.5, 0.0, T_bw)
x0_turbid_seed = find_equilibria_from_initial_conditions(L=1.5, F=F_sweep, p=params)[1]
sweep_bw = sweep_simulate(L_bw, x0_turbid_seed, F_sweep, params,
                          window_len=window_length, step=step_size, noise_std=noise_std, seed=0, dt=0.1, additive_noise=False)

traj_fw, Lc_fw, ms_fw = sweep_fw['traj'], sweep_fw['mean_Ls'], sweep_fw['mean_states']
traj_bw, Lc_bw, ms_bw = sweep_bw['traj'], sweep_bw['mean_Ls'], sweep_bw['mean_states']

In [ ]:
fig_traj = build_trajectory_figure(sweep_fw, sweep_bw)
fig_traj.show()

In [ ]:
L_tipp_fw = 1.3
L_tipp_bw = 0.512

In [ ]:
# # Sweep analytical decomposition
# analytical_decomp_fw = sweep_decompose_analytical(sweep_fw, params, T_max=T_max)
# analytical_decomp_bw = sweep_decompose_analytical(sweep_bw, params, T_max=T_max)

In [ ]:
# # Sweep causal effect decomposition
# ce_decomp_fw = sweep_decompose_causaleffect(sweep_fw, T_max=T_max, detrend_window=window_length)
# ce_decomp_bw = sweep_decompose_causaleffect(sweep_bw, T_max=T_max, detrend_window=window_length)

These are saved results as it takes quite some time to run:

In [ ]:
# _analytical_keys = ['ts', 'A_total', 'A_direct', 'A_nutrient', 'A_trophic', 'A_mixed', 
#                     'A_nutrient',
#              'A_eq', 'J']
# _ce_keys = ['ts', 'A_total', 'A_direct', 'A_trophic', 'A_mixed', 
#             'A_nutrient',
#              ]

# _r = np.load('sweep_results_both__Tmax100days_wl1000_step10_F001_T365times50_noM.npz')
# traj_fw, Lc_fw, ms_fw = _r['traj_fw'], _r['Lc_fw'], _r['ms_fw']
# traj_bw, Lc_bw, ms_bw = _r['traj_bw'], _r['Lc_bw'], _r['ms_bw']

# n_fw, n_bw = len(Lc_fw), len(Lc_bw)
# analytical_decomp_fw = [{k: _r[f'dfw_{k}'][i] for k in _analytical_keys} for i in range(n_fw)]
# analytical_decomp_bw = [{k: _r[f'dbw_{k}'][i] for k in _analytical_keys} for i in range(n_bw)]
# ce_decomp_fw = [{k: _r[f'tfw_{k}'][i] for k in _ce_keys} for i in range(n_fw)]
# ce_decomp_bw = [{k: _r[f'tbw_{k}'][i] for k in _ce_keys} for i in range(n_bw)]

## Sweep heatmaps

In [ ]:
def signed_log(x, eps=1e-3):
    x = np.asarray(x, float)
    return np.sign(x) * np.log10(1.0 + np.abs(x) / eps)

In [ ]:
def sweep_heatmap(Lc_fw, Lc_bw, analytical_decomp_fw=None, analytical_decomp_bw=None, ce_decomp_fw=None, ce_decomp_bw=None,
                  source='causaleffect', T_show=30, clip_quantile=0.99,
                  tipp_fw=None, tipp_bw=None, window_L=None, L_range=None, mechs=None, transform=lambda x: x):
    
    if mechs is None:
        mechs = [('A_total', 'Total'), ('A_direct', 'Direct'), ('A_nutrient', 'Nutrient'), ('A_trophic', 'Trophic'), ('A_mixed', 'Mixed')
                 ]
    fw_list = ce_decomp_fw if source == 'causaleffect' else analytical_decomp_fw
    bw_list = ce_decomp_bw if source == 'causaleffect' else analytical_decomp_bw

    ts = next(d for d in fw_list if d is not None)['ts']
    keep = ts >= 0 if T_show is None else (ts >= 0) & (ts <= T_show)
    ts_keep = ts[keep]

    def stack(dec_list, key):
        M = np.full((len(ts), len(dec_list)), np.nan)
        for i, d in enumerate(dec_list):
            if d is not None:
                M[:, i] = d[key]
        return M[keep]

    n_rows, vsp = len(mechs), 0.06
    titles = [t for (_, name) in mechs
              for t in (f'{name} — forward (L↑)', f'{name} — backward (L↓)')]
    fig = make_subplots(rows=n_rows, cols=2, subplot_titles=titles,
                        shared_xaxes='columns', shared_yaxes='rows',
                        horizontal_spacing=0.07, vertical_spacing=vsp)

    sub_h = (1.0 - (n_rows - 1) * vsp) / n_rows


    for r, (key, name) in enumerate(mechs, start=1):
        fw_M, bw_M = stack(fw_list, key), stack(bw_list, key)
        fw_M, bw_M = transform(fw_M), transform(bw_M)
        allv = np.abs(np.concatenate([fw_M.ravel(), bw_M.ravel()]))
        allv = allv[np.isfinite(allv)]
        vmax = np.quantile(allv, clip_quantile) if len(allv) else 1.0
        vmax = vmax if vmax > 0 else 1.0
        y_center = 1 - (r - 0.5) * sub_h - (r - 1) * vsp
        for c, (M, Lc) in enumerate([(fw_M, Lc_fw), (bw_M, Lc_bw)], start=1):
            order = np.argsort(Lc)            # heatmaps need x ascending; the backward
            M, Lc = M[:, order], np.asarray(Lc, float)[order]
            if L_range is not None: 
                pad = np.full((M.shape[0], 1), np.nan)
                M = np.hstack([pad, M, pad])                     
                Lc = np.concatenate([[L_range[0]], Lc, [L_range[1]]])
            fig.add_trace(
                go.Heatmap(z=M, x=Lc, y=ts_keep, colorscale='RdBu_r',
                           zmin=-vmax, zmax=vmax, zmid=0, showscale=(c == 2),
                           colorbar=dict(title=name, len=sub_h * 0.9, y=y_center,
                                         yanchor='middle', x=1.02, thickness=10,
                                         tickfont=dict(size=8))),
                row=r, col=c)
        fig.update_yaxes(title_text='T (days)', row=r, col=1)
        fig.update_yaxes(range=[float(ts_keep.min()) - 0.5, float(ts_keep.max()) + 0.5], row=r, col=1)
        fig.update_yaxes(range=[float(ts_keep.min()) - 0.5, float(ts_keep.max()) + 0.5], row=r, col=2)
        fig.update_yaxes(title_text='T (days)', row=r, col=1)


    y0b, y1b = float(ts_keep.min()) - 0.5, float(ts_keep.max()) + 0.5   # band spans the lag axis
    for col, L_t in [(1, tipp_fw), (2, tipp_bw)]:
        if L_t is None:
            continue
        for r in range(1, n_rows + 1):
            if window_L:
                x0b, x1b = L_t - window_L / 2, L_t + window_L / 2
                fig.add_trace(
                    go.Scatter(x=[x0b, x1b, x1b, x0b, x0b], y=[y0b, y0b, y1b, y1b, y0b],
                               mode='lines', line=dict(width=0), fillcolor='rgba(0,0,0,0)',
                               fill='toself', fillpattern=dict(shape='/', fgcolor='grey',
                                   bgcolor='rgba(0,0,0,0)', size=8, solidity=0.3),
                               showlegend=False, hoverinfo='skip'),
                    row=r, col=col)
            fig.add_vline(x=L_t, line=dict(color='grey', width=1.2, dash='dot'),
                          row=r, col=col)

    fig.update_xaxes(title_text='L<sub>c</sub>', row=n_rows, col=1)
    fig.update_xaxes(title_text='L<sub>c</sub>', row=n_rows, col=2)
    for ann in fig.layout.annotations:
        ann.font = dict(size=11)
    fig.update_layout(height=180 * n_rows, width=1000, template='plotly_white',
                      margin=dict(r=110, t=70),)
    return fig

In [ ]:
# window_length = 1000
window_L = window_length * abs(L_fw[-1] - L_fw[0]) / (len(L_fw) - 1)

fig_sweep_heatmaps_datadriven = sweep_heatmap(Lc_fw=Lc_fw, Lc_bw=Lc_bw, 
                                              ce_decomp_fw=ce_decomp_fw, ce_decomp_bw=ce_decomp_bw, 
                                              source='causaleffect', T_show=150, clip_quantile=0.98, 
                                              tipp_fw=L_tipp_fw, tipp_bw=L_tipp_bw, window_L=window_L, 
                                              L_range=(float(min(L_fw.min(), L_bw.min())), float(max(L_fw.max(), L_bw.max()))),
                                              transform=lambda x: signed_log(x, eps=1e-3))
fig_sweep_heatmaps_datadriven.show()

In [ ]:
window_L = window_length * abs(L_fw[-1] - L_fw[0]) / (len(L_fw) - 1)

fig_sweep_heatmaps_analytical = sweep_heatmap(Lc_fw=Lc_fw, Lc_bw=Lc_bw,
                                              analytical_decomp_fw=analytical_decomp_fw, analytical_decomp_bw=analytical_decomp_bw,
                                              source='analytical', T_show=100, clip_quantile=0.98,
                                              tipp_fw=L_tipp_fw, tipp_bw=L_tipp_bw, window_L=window_L,
                                              L_range=(float(min(L_fw.min(), L_bw.min())), float(max(L_fw.max(), L_bw.max()))),
                                              transform=lambda x: signed_log(x, eps=1e-3))
fig_sweep_heatmaps_analytical.show()